# AI Foundry Key Verification

이 노트북은 `deploy_all.sh`가 생성한 Excel 파일(`output/team_keys.xlsx`)의 API 키가 정상 작동하는지 검증합니다.

## 인증 방식

이 샘플은 **API 키 인증**(`api-key` 헤더)만 사용합니다.  
외부 개발자가 Entra ID 없이 Resource Key만으로 Azure AI Services 엔드포인트에 접근하는 시나리오를 전제합니다.

> **PoC 전용**: 프로덕션 환경에서는 Azure Key Vault + Entra ID(DefaultAzureCredential) 사용을 권장합니다.  
> Excel에 평문으로 기록된 키는 샘플/PoC 목적이며, 절대로 Git에 커밋하지 마십시오.

In [ ]:
# --- Dependencies ---
%pip install openpyxl requests --quiet

import openpyxl
import requests
from pathlib import Path

In [ ]:
# --- Configuration ---
# Excel 파일 경로 (deploy_all.sh 실행 후 생성됨)
EXCEL_PATH = Path("../output/team_keys.xlsx")

# Chat Completion에 사용할 모델 배포 이름 (예: gpt-4o)
CHAT_DEPLOYMENT = "gpt-4o"

# Embedding에 사용할 모델 배포 이름 (예: text-embedding-3-large)
EMBEDDING_DEPLOYMENT = "text-embedding-3-large"

# Azure OpenAI API 버전
API_VERSION = "2024-12-01-preview"

In [ ]:
# --- Read Excel & Display Teams ---
if not EXCEL_PATH.exists():
    raise FileNotFoundError(
        f"{EXCEL_PATH} not found. Run './scripts/deploy_all.sh' first to generate the key Excel."
    )

wb = openpyxl.load_workbook(EXCEL_PATH, read_only=True)
ws = wb.active

# Parse rows (skip header)
rows = list(ws.iter_rows(min_row=2, values_only=True))
wb.close()

if not rows:
    raise ValueError("Excel file has no data rows.")

# Display available teams/regions
print(f"{'#':<4} {'Team':<12} {'Region':<18} {'Endpoint':<55} {'Models'}")
print("-" * 120)
for idx, row in enumerate(rows):
    team, _sub, _rg, region, endpoint, _k1, _k2, models = row
    print(f"{idx:<4} {team:<12} {region:<18} {endpoint:<55} {models}")

print(f"\n총 {len(rows)}개 Foundry Resource 발견")

In [ ]:
# --- Select Row & Test Chat Completion ---
# 테스트할 행 번호를 선택하세요 (위 테이블의 # 열 참고)
SELECTED_ROW = 0

row = rows[SELECTED_ROW]
team, _sub, _rg, region, endpoint, key1, _key2, models = row

print(f"Team: {team} | Region: {region}")
print(f"Endpoint: {endpoint}")
print(f"Models: {models}")
print()

# Chat Completion 호출
url = f"{endpoint.rstrip('/')}/openai/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
headers = {
    "Content-Type": "application/json",
    "api-key": key1,
}
payload = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say 'Key verification successful!' in one sentence."},
    ],
    "max_tokens": 50,
}

print(f"Calling {CHAT_DEPLOYMENT}...")
resp = requests.post(url, headers=headers, json=payload, timeout=30)
resp.raise_for_status()

result = resp.json()
message = result["choices"][0]["message"]["content"]
usage = result.get("usage", {})

print(f"Response: {message}")
print(f"Tokens — prompt: {usage.get('prompt_tokens')}, completion: {usage.get('completion_tokens')}, total: {usage.get('total_tokens')}")
print("\n✅ Chat Completion 키 검증 성공!")

In [ ]:
# --- Test Embedding ---
# 선택한 Foundry Resource에 text-embedding 모델이 배포되어 있어야 합니다.
if EMBEDDING_DEPLOYMENT not in (models or ""):
    print(f"⚠️  '{EMBEDDING_DEPLOYMENT}' 모델이 '{team}/{region}'에 배포되어 있지 않습니다. 건너뜁니다.")
else:
    emb_url = f"{endpoint.rstrip('/')}/openai/deployments/{EMBEDDING_DEPLOYMENT}/embeddings?api-version={API_VERSION}"
    emb_payload = {
        "input": "Azure AI Foundry cost governance sample",
    }

    print(f"Calling {EMBEDDING_DEPLOYMENT}...")
    emb_resp = requests.post(emb_url, headers=headers, json=emb_payload, timeout=30)
    emb_resp.raise_for_status()

    emb_result = emb_resp.json()
    embedding = emb_result["data"][0]["embedding"]
    emb_usage = emb_result.get("usage", {})

    print(f"Embedding dimensions: {len(embedding)}")
    print(f"First 5 values: {embedding[:5]}")
    print(f"Tokens — prompt: {emb_usage.get('prompt_tokens')}, total: {emb_usage.get('total_tokens')}")
    print("\n✅ Embedding 키 검증 성공!")

## 다음 단계

키 검증이 완료되었습니다. 다음으로:

1. **Cost Dashboard 확인**: Azure Portal → 각 Team Subscription → Application Insights → Workbook에서 토큰 사용량과 비용을 확인합니다.
2. **Budget Alert**: Terraform이 배포한 Budget Alert가 설정한 임계값을 초과하면 `alert_email`로 알림이 발송됩니다.
3. **프로덕션 전환 시**: API 키 대신 Azure Key Vault + Entra ID(`DefaultAzureCredential`) 기반 인증으로 전환하세요.

---

자세한 내용은 [README.md](../README.md)를 참고하세요.